# ARDL long-run multipliers

Two level relationships. Fig. 2(b), right, is the 31-day exposure-tier share. Fig. 2(d) is the elasticity of overall engagement depth. Bounds, the t statistic on rho, Breusch-Godfrey, CUSUM, CUSUMSQ, and the lag grid are in `analysis/robustness/timeseries/ardl.ipynb`.

### 1. Exposure-tier shares

Each outcome is the share of platform-labeled videos uploaded that day whose 31-day exposure falls in one tier. The regressors are labeled supply and unlabeled supply.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

import pandas as pd
from core.columns import EXPOSURE_SHARES, LABELED_SUPPLY, UNLABELED_SUPPLY
from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
daily = ensure_synthetic(root, cfg, rebuild=False)["daily_panel"]
tiers = daily[list(EXPOSURE_SHARES) + [LABELED_SUPPLY, UNLABELED_SUPPLY]]
print(len(tiers), "days")
tiers.head(8)


180 days


,share_le_10,share_11_to_100,share_101_to_1000,share_above_1000,labeled_supply,unlabeled_supply
0,0.249373,0.347010,0.245901,0.157716,260.000000,894.285721
1,0.115610,0.364517,0.405341,0.114532,249.476859,899.379997
2,0.157602,0.319486,0.384278,0.138635,249.702765,900.793579
3,0.049113,0.320295,0.521773,0.108819,226.287062,900.525169
4,0.037443,0.238056,0.629138,0.095363,222.256079,897.671212
5,0.048599,0.281322,0.555419,0.114661,220.198807,905.886064
6,0.108038,0.287033,0.481730,0.123198,238.781954,910.630683
7,0.087722,0.309671,0.461440,0.141168,228.877460,920.188495


### 2. Error-correction regressors

The level equation is rewritten so the lagged level of the outcome and the lagged levels of the two supplies carry the long-run coefficients. The example is the lowest tier. theta = -pi / rho.

In [2]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.ardl import error_correction_frame

work = error_correction_frame(
    daily,
    EXPOSURE_SHARES[0],
    [LABELED_SUPPLY, UNLABELED_SUPPLY],
    cfg["time_exog"],
    cfg["ardl_p"],
    cfg["ardl_q"],
)
level_columns = ["dy", "y_l1", f"{LABELED_SUPPLY}_l1", f"{UNLABELED_SUPPLY}_l1"]
print(len(work), "rows;", ", ".join(level_columns))
work[level_columns].head(8)


179 rows; dy, y_l1, labeled_supply_l1, unlabeled_supply_l1


,dy,y_l1,labeled_supply_l1,unlabeled_supply_l1
0,-0.133763,0.249373,260.000000,894.285721
1,0.041991,0.115610,249.476859,899.379997
2,-0.108489,0.157602,249.702765,900.793579
3,-0.011670,0.049113,226.287062,900.525169
4,0.011156,0.037443,222.256079,897.671212
5,0.059440,0.048599,220.198807,905.886064
6,-0.020317,0.108038,238.781954,910.630683
7,-0.007396,0.087722,228.877460,920.188495


### 3. Tier multipliers

One equation per tier. The labeled-supply row is the association reported in the text. Unlabeled supply is the control in the same equation.

In [3]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import run_tiers_notebook

keep = [
    "outcome", "regressor", "p", "q", "rho", "pi", "theta",
    "theta_standard_error", "theta_ci95_low", "theta_ci95_high",
]
run_tiers_notebook()[keep]


,outcome,regressor,p,q,rho,pi,theta,theta_standard_error,theta_ci95_low,theta_ci95_high
0,share_le_10,labeled_supply,1,1,-0.832599,0.004906,0.005892,0.000792,0.004339,0.007444
1,share_le_10,unlabeled_supply,1,1,-0.832599,0.000260,0.000313,0.000301,-0.000278,0.000903
2,share_11_to_100,labeled_supply,1,1,-0.954445,0.001363,0.001428,0.000817,-0.000172,0.003029
3,share_11_to_100,unlabeled_supply,1,1,-0.954445,0.000128,0.000134,0.000371,-0.000594,0.000861
4,share_101_to_1000,labeled_supply,1,1,-1.087739,-0.007852,-0.007219,0.000555,-0.008306,-0.006131
5,share_101_to_1000,unlabeled_supply,1,1,-1.087739,-0.000769,-0.000707,0.000253,-0.001202,-0.000212
6,share_above_1000,labeled_supply,1,1,-0.971632,-0.000008,-0.000008,0.000272,-0.000540,0.000524
7,share_above_1000,unlabeled_supply,1,1,-0.971632,0.000279,0.000288,0.000127,0.000038,0.000537


### 4. Engagement depth

The outcomes are the log overall full-view rate and the log overall valid-view rate. The regressors are log labeled supply and log unlabeled supply.

In [4]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.columns import LOG_FULL_VIEW_RATE, LOG_LABELED_SUPPLY, LOG_UNLABELED_SUPPLY, LOG_VALID_VIEW_RATE

depth = daily[[LOG_FULL_VIEW_RATE, LOG_VALID_VIEW_RATE, LOG_LABELED_SUPPLY, LOG_UNLABELED_SUPPLY]]
print(len(depth), "days")
depth.head(8)


180 days


,log_full_view_rate,log_valid_view_rate,log_labeled_supply,log_unlabeled_supply
0,-2.120264,-1.078810,5.560682,6.796025
1,-2.106411,-1.071694,5.519366,6.801706
2,-2.120849,-1.080307,5.520271,6.803276
3,-2.119323,-1.076834,5.421804,6.802978
4,-2.120348,-1.089045,5.403830,6.799804
5,-2.116941,-1.079398,5.394531,6.808914
6,-2.117461,-1.081343,5.475551,6.814137
7,-2.118875,-1.070922,5.433187,6.824579


### 5. Elasticities

theta on log labeled supply is the long-run elasticity of engagement depth. Log unlabeled supply is the control.

In [5]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import run_engagement_notebook

run_engagement_notebook()[keep]


,outcome,regressor,p,q,rho,pi,theta,theta_standard_error,theta_ci95_low,theta_ci95_high
0,log_full_view_rate,log_labeled_supply,1,1,-0.410003,0.060994,0.148764,0.098461,-0.044219,0.341748
1,log_full_view_rate,log_unlabeled_supply,1,1,-0.410003,0.003378,0.008239,0.169127,-0.323250,0.339728
2,log_valid_view_rate,log_labeled_supply,1,1,-0.532000,0.005533,0.010401,0.060198,-0.107588,0.128390
3,log_valid_view_rate,log_unlabeled_supply,1,1,-0.532000,0.073930,0.138967,0.124571,-0.105192,0.383126
